In [15]:
%pip -q install requests lxml cssselect


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import requests
from lxml import html
from urllib.parse import urljoin, unquote

def scrape_quests_api():
    """Use Fandom's MediaWiki API to get quest page content"""
    api_url = "https://arcanum.fandom.com/api.php"
    
    params = {
        "action": "parse",
        "page": "Quests",
        "prop": "text",
        "format": "json"
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    try:
        response = requests.get(api_url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        
        html_content = data['parse']['text']['*']
        tree = html.fromstring(html_content)
        
        # Skill quests: td:last-child > a
        skill_elements = tree.cssselect('td:last-child > a')
        skill_urls = []
        for elem in skill_elements:
            href = elem.get('href')
            if href and '/wiki/' in href:
                full_url = urljoin("https://arcanum.fandom.com", href.split('?')[0])
                skill_urls.append(full_url)
        
        # Other quests: //li//a[@href[starts-with(.,"/wiki")]]
        other_elements = tree.xpath('//li//a[@href[starts-with(.,"/wiki")]]')
        other_urls = []
        for elem in other_elements:
            href = elem.get('href')
            if href:
                full_url = urljoin("https://arcanum.fandom.com", href.split('?')[0])
                if full_url not in skill_urls:
                    other_urls.append(full_url)
        
        # Remove duplicates
        seen = set()
        skill_urls = [x for x in skill_urls if not (x in seen or seen.add(x))]
        other_urls = [x for x in other_urls if not (x in seen or seen.add(x))]
        
        return {
            'skill_quests': skill_urls,
            'other_quests': other_urls,
            'total': len(skill_urls) + len(other_urls)
        }
        
    except Exception as e:
        print(f"Error: {e}")
        return None


In [18]:
result = scrape_quests_api()

if result:
    print(f"=== Skill Quests ({len(result['skill_quests'])}) ===")
    for url in result['skill_quests']:
        print(url)
        
    print(f"\n=== Other Quests ({len(result['other_quests'])}) ===")
    for url in result['other_quests']:
        print(url)

=== Skill Quests (16) ===
https://arcanum.fandom.com/wiki/Find_the_Bow_of_Ecclesiastes
https://arcanum.fandom.com/wiki/Kill_Sir_Garrick_Stout
https://arcanum.fandom.com/wiki/Find_Lady_Druella
https://arcanum.fandom.com/wiki/Retrieve_Azram%27s_Star
https://arcanum.fandom.com/wiki/Find_the_Master_of_Backstab
https://arcanum.fandom.com/wiki/Run_Around_Tarant_in_Your_Underwear
https://arcanum.fandom.com/wiki/Find_the_Master_of_Prowling
https://arcanum.fandom.com/wiki/Get_staff_of_K%E2%80%99an_T%E2%80%99au
https://arcanum.fandom.com/wiki/Gamble_with_Gurin_Rockharrow
https://arcanum.fandom.com/wiki/Acquire_Ten_Thousand_Gold_Pieces
https://arcanum.fandom.com/wiki/Find_the_Master_of_Heal
https://arcanum.fandom.com/wiki/Negotiations_with_Caladon
https://arcanum.fandom.com/wiki/Find_Proof_that_Maxim%27s_Air_Machines_Flew
https://arcanum.fandom.com/wiki/Rescue_Mrs._Rolland_Unharmed
https://arcanum.fandom.com/wiki/Free_J.T._Morgan
https://arcanum.fandom.com/wiki/Survive_the_Training_Maze

=== Othe

In [20]:
import requests
import csv
import time
from lxml import html
from urllib.parse import urljoin, urlparse, unquote
import random

def fetch_page_content(page_name):
    """Fetch page content via MediaWiki API with proper error handling"""
    api_url = "https://arcanum.fandom.com/api.php"
    params = {
        "action": "parse",
        "page": page_name,
        "prop": "text",
        "format": "json",
        "redirects": 1  # Follow redirects
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(api_url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Check for API errors
        if 'error' in data:
            print(f"  API Error for '{page_name}': {data['error'].get('info', 'Unknown error')}")
            return None
            
        if 'parse' not in data:
            print(f"  No 'parse' key in response for '{page_name}'. Response keys: {list(data.keys())}")
            return None
            
        return data['parse']['text']['*']
        
    except requests.exceptions.HTTPError as e:
        print(f"  HTTP Error: {e}")
        return None
    except Exception as e:
        print(f"  Error: {e}")
        return None

def extract_sections(page_url, html_content):
    """
    Extract parts starting with h2.
    """
    if not html_content:
        return []
        
    tree = html.fromstring(html_content)
    sections = []
    
    # Find content container
    content_div = tree.xpath('//div[@class="mw-parser-output"]')
    if not content_div:
        print(f"  No mw-parser-output found")
        return []
    
    content_div = content_div[0]
    h2_elements = content_div.xpath('.//h2[not(ancestor::table) and not(ancestor::infobox)]')
    
    print(f"  Found {len(h2_elements)} h2 sections")
    
    for i, h2 in enumerate(h2_elements):
        # Clean h2 text (remove [edit] spans)
        for span in h2.xpath('.//span[@class="mw-editsection"]'):
            span.drop_tree()
        h2_text = h2.text_content().strip()
        
        # Get all elements between this h2 and the next h2
        section_elements = []
        current = h2.getnext()
        while current is not None and current.tag != 'h2':
            # Skip navigation boxes, tables, etc.
            if current.tag not in ['table', 'nav', 'footer']:
                section_elements.append(current)
            current = current.getnext()
        
        # Look for h3 subdivisions
        h3_elements = [el for el in section_elements if el.tag == 'h3']
        
        section_texts = []
        for el in section_elements:
            if el.tag not in ['table', 'nav']:
                text = el.text_content().strip()
                if text:
                    section_texts.append(text)
        
        full_text = ' '.join(section_texts)
        if full_text and len(full_text) > 20:
            sections.append({
                'LINK': page_url,
                'TITLE': h2_text,
                'TEXT': full_text.replace('\n', ' ').replace('\r', ' ')
            })
    
    return sections

def get_quest_urls_from_page():
    """Fetch real quest URLs from the Quests page"""
    html_content = fetch_page_content("Quests")
    if not html_content:
        return []
    
    tree = html.fromstring(html_content)
    urls = []
    
    # Try your selectors from the original request
    # Skill quests: td:last-child > a
    skill_elements = tree.cssselect('td:last-child > a')
    for elem in skill_elements:
        href = elem.get('href')
        if href and '/wiki/' in href:
            clean_href = href.split('?')[0]
            full_url = urljoin("https://arcanum.fandom.com", clean_href)
            urls.append(full_url)
    
    # Other quests: //li//a[@href[starts-with(.,"/wiki")]]
    other_elements = tree.xpath('//li//a[@href[starts-with(.,"/wiki")]]')
    for elem in other_elements:
        href = elem.get('href')
        if href:
            clean_href = href.split('?')[0]
            full_url = urljoin("https://arcanum.fandom.com", clean_href)
            urls.append(full_url)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_urls = []
    for url in urls:
        if url not in seen and 'Quests' not in url:  # Exclude the main Quests page
            seen.add(url)
            unique_urls.append(url)
    
    print(f"Found {len(unique_urls)} unique quest URLs from Quests page")
    return unique_urls

def process_quests(quest_urls, output_file='arcanum_quests.csv'):
    """Process list of quest URLs and save to CSV"""
    all_data = []
    
    for idx, url in enumerate(quest_urls, 1):
        print(f"\n[{idx}/{len(quest_urls)}] Processing: {url}")
        
        # Extract page name from URL
        parsed = urlparse(url)
        page_name = unquote(parsed.path.split('/')[-1])
        if not page_name:
            print(f"  Skipping: empty page name")
            continue
        
        # Get HTML via API
        html_content = fetch_page_content(page_name)
        if not html_content:
            continue
            
        # Extract sections
        sections = extract_sections(url, html_content)
        if sections:
            print(f"  Extracted {len(sections)} sections")
            all_data.extend(sections)
        else:
            print(f"  No sections found")
        
        # Rate limiting
        time.sleep(random.uniform(1, 2))
    
    # Write to CSV
    if all_data:
        with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=['LINK', 'TITLE', 'TEXT'])
            writer.writeheader()
            writer.writerows(all_data)
        print(f"\n✓ Saved {len(all_data)} sections to {output_file}")
    else:
        print("\n✗ No data extracted")

if __name__ == "__main__":
    # Option 1: Use real quest URLs fetched from the Quests page
    print("Fetching quest list from Quests page...")
    quest_urls = get_quest_urls_from_page()
    
    # Option 2: Or manually provide specific quest URLs
    # quest_urls = [
    #     "https://arcanum.fandom.com/wiki/Find_the_Stillwater_Giant",
    #     "https://arcanum.fandom.com/wiki/Rescue_the_Kidnapped_Girl",  # This might not exist
    # ]
    
    if quest_urls:
        process_quests(quest_urls)
    else:
        print("No quest URLs found. Check if the Quests page structure has changed.")

Fetching quest list from Quests page...
Found 114 unique quest URLs from Quests page

[1/114] Processing: https://arcanum.fandom.com/wiki/Find_the_Bow_of_Ecclesiastes
  No mw-parser-output found
  No sections found

[2/114] Processing: https://arcanum.fandom.com/wiki/Kill_Sir_Garrick_Stout
  No mw-parser-output found
  No sections found

[3/114] Processing: https://arcanum.fandom.com/wiki/Find_Lady_Druella
  No mw-parser-output found
  No sections found

[4/114] Processing: https://arcanum.fandom.com/wiki/Retrieve_Azram%27s_Star
  No mw-parser-output found
  No sections found

[5/114] Processing: https://arcanum.fandom.com/wiki/Find_the_Master_of_Backstab
  No mw-parser-output found
  No sections found

[6/114] Processing: https://arcanum.fandom.com/wiki/Run_Around_Tarant_in_Your_Underwear
  No mw-parser-output found
  No sections found

[7/114] Processing: https://arcanum.fandom.com/wiki/Find_the_Master_of_Prowling
  No mw-parser-output found
  No sections found


KeyboardInterrupt: 

In [ ]:
import requests
import csv
import time
from lxml import html, etree
from urllib.parse import urljoin, urlparse, unquote
import random

def fetch_page_content(page_name):
    """Fetch page content via MediaWiki API"""
    api_url = "https://arcanum.fandom.com/api.php"
    params = {
        "action": "parse",
        "page": page_name,
        "prop": "text",
        "format": "json",
        "redirects": 1
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(api_url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if 'error' in data:
            print(f"  API Error: {data['error'].get('info', 'Unknown')}")
            return None
            
        return data['parse']['text']['*']
        
    except Exception as e:
        print(f"  Error: {e}")
        return None

def is_element(node):
    """Check if node is an element"""
    return hasattr(node, 'tag') and not isinstance(node, (etree._Comment, etree._ProcessingInstruction))

def clean_text(elem):
    """Extract clean text from an element without duplication"""
    if not is_element(elem):
        return ""
    
    # Work on a copy so we don't modify the original tree
    import copy
    temp = copy.deepcopy(elem)
    
    # Remove unwanted elements
    for selector in ['.//script', './/style', './/nav', './/span[@class="mw-editsection"]']:
        for bad in temp.xpath(selector):
            bad.getparent().remove(bad)
    
    # text_content() extracts all text recursively without duplication
    text = temp.text_content()
    
    # Clean up whitespace
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return ' '.join(lines)

def get_content_root(tree):
    """Find main content container"""
    result = tree.xpath('//div[@class="mw-parser-output"]')
    if result:
        return result[0]
    
    result = tree.xpath('//div[@id="mw-content-text"]')
    if result:
        inner = result[0].xpath('.//div[@class="mw-parser-output"]')
        if inner:
            return inner[0]
        return result[0]
    
    if tree.tag == 'div' and 'mw-parser-output' in (tree.get('class') or ''):
        return tree
    
    return tree

def get_next_element_sibling(elem):
    """Get next sibling that is an element"""
    current = elem.getnext()
    while current is not None and not is_element(current):
        current = current.getnext()
    return current

def extract_sections(page_url, html_content):
    """Extract h2 sections with h3 subdivisions"""
    
    if not html_content:
        return []
    
    tree = html.fromstring(html_content)
    root = get_content_root(tree)
        
    if root is None:
        return []
    
    sections = []
    
    # Iterate through all elements in root
    children = list(root)
    i = 0
    while i < len(children):
        elem = children[i]
        
        if not is_element(elem) or elem.tag != 'h2':
            i += 1
            continue
        
        # Clean h2 title
        h2_title = clean_text(elem)
        
        if not h2_title:
            i += 1
            continue
        
        # Collect siblings until next h2
        siblings = []
        j = i + 1
        while j < len(children):
            sibling = children[j]
            if is_element(sibling) and sibling.tag == 'h2':
                break
            if is_element(sibling):
                siblings.append(sibling)
            j += 1
        
        content_parts = []
        for el in siblings:
            if el.tag not in ['table', 'nav', 'script', 'style']:
                txt = clean_text(el)
                if txt:
                    content_parts.append(txt)
        
        full_text = ' '.join(content_parts)
        if full_text and len(full_text) > 20:
            sections.append({
                'LINK': page_url,
                'TITLE': h2_title,
                'TEXT': full_text.strip()
            })
        
        i = j  # Skip to after this section
    
    return sections

def get_quest_urls():
    """Fetch quest URLs from Quests page"""
    html_content = fetch_page_content("Quests")
    if not html_content:
        return []
    
    tree = html.fromstring(html_content)
    urls = []
    
    for elem in tree.cssselect('td:last-child > a'):
        href = elem.get('href')
        if href and '/wiki/' in href:
            clean = href.split('?')[0].split('#')[0]
            url = urljoin("https://arcanum.fandom.com", clean)
            if url not in urls and 'Quests' not in url:
                urls.append(url)
    
    for elem in tree.xpath('//li//a[@href[starts-with(.,"/wiki")]]'):
        href = elem.get('href')
        if href:
            clean = href.split('?')[0].split('#')[0]
            url = urljoin("https://arcanum.fandom.com", clean)
            if url not in urls and 'Quests' not in url:
                urls.append(url)
    
    print(f"Found {len(urls)} unique quest URLs")
    return urls

def process_quests(quest_urls, output_file='arcanum_quests.csv'):
    all_data = []
    
    for idx, url in enumerate(quest_urls, 1):
        print(f"\n[{idx}/{len(quest_urls)}] {url}")
        
        parsed = urlparse(url)
        page_name = unquote(parsed.path.split('/')[-1])
        if not page_name:
            continue
        
        html_content = fetch_page_content(page_name)
        if not html_content:
            continue
        
        sections = extract_sections(url, html_content)
        if sections:
            print(f"  ✓ {len(sections)} sections")
            all_data.extend(sections)
        else:
            print(f"  ✗ No sections")
        
        time.sleep(random.uniform(0.3, 0.8))
    
    if all_data:
        with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=['LINK', 'TITLE', 'TEXT'])
            writer.writeheader()
            writer.writerows(all_data)
        print(f"\n✓ Saved {len(all_data)} sections to {output_file}")
    else:
        print("\n✗ No data extracted")

if __name__ == "__main__":
    print("Fetching quest list...")
    urls = get_quest_urls()
    
    if urls:
        process_quests(urls[:10])
    else:
        print("No URLs found")

Fetching quest list...
Found 114 unique quest URLs

[1/10] https://arcanum.fandom.com/wiki/Find_the_Bow_of_Ecclesiastes
  ✓ 2 sections

[2/10] https://arcanum.fandom.com/wiki/Kill_Sir_Garrick_Stout
  ✓ 1 sections

[3/10] https://arcanum.fandom.com/wiki/Find_Lady_Druella
  ✓ 1 sections

[4/10] https://arcanum.fandom.com/wiki/Retrieve_Azram%27s_Star
  ✓ 1 sections

[5/10] https://arcanum.fandom.com/wiki/Find_the_Master_of_Backstab
  ✗ No sections

[6/10] https://arcanum.fandom.com/wiki/Run_Around_Tarant_in_Your_Underwear
  ✗ No sections

[7/10] https://arcanum.fandom.com/wiki/Find_the_Master_of_Prowling
  ✗ No sections

[8/10] https://arcanum.fandom.com/wiki/Get_staff_of_K%E2%80%99an_T%E2%80%99au
  ✗ No sections

[9/10] https://arcanum.fandom.com/wiki/Gamble_with_Gurin_Rockharrow
  ✓ 3 sections

[10/10] https://arcanum.fandom.com/wiki/Acquire_Ten_Thousand_Gold_Pieces
  ✗ No sections

✓ Saved 8 sections to arcanum_quests.csv


In [ ]:
# TODO: hanlde no sections: div.mw-parser-output
# TODO: Walkthrough => Gamble with Gurin Rockharrow: Walkthrough